# Master pipeline — Time Headway distribution & covariate modelling (data3)

Runs **step by step**; each numbered step writes its own Excel so you can stop and inspect.

1. Inferential statistics (Overall / BTW / PR, kept separate) → **Excel 1**
2. Distribution fitting per pair (all 10 pairs, all candidates) → **Excel 2**
3. Pick the single best overall family across pairs (report all; expected Weibull)
4. Drop low-sample pairs (editable) — keeps 7 pairs
5. Covariate screening per kept pair under the **overall family** → **Excel 3**
6. Covariate screening under each pair's **own best distribution** (pairs that don't take Weibull) → **Excel 4**
7. Finalize the covariate set per pair (strict rule + de-clustering) → **Excel 5**

Column handling: subject column and speed column are auto-detected (`V_Target`/`V_Subject`,
`Target_Speed_km/hr`/`Subject_Speed_km/hr`); `share_bus` used only if present.
AFT models fix `loc = 0` so covariate likelihood-ratio tests are nested/valid; marginal
distribution fitting (Step 2) uses a free `loc`.

In [1]:
# --- Cell 1: Imports, paths, configuration ---
import os, warnings
import numpy as np
import pandas as pd
from scipy import stats, optimize
warnings.simplefilter("ignore")

BASE      = r"D:\Headway"
DATA_PATH = os.path.join(BASE, "data3.xlsx")
TABLES    = os.path.join(BASE, "Tables")
os.makedirs(TABLES, exist_ok=True)

OUTCOME       = "Time_Headway"
ALPHA         = 0.05
DAIC_MIN      = 2.0     # strict-improvement AIC threshold
MIN_BIN_GROUP = 5       # min count per binary group to fit a covariate
MIN_N_FIT     = 30      # pairs below this are flagged as small for distribution fitting

CANDIDATE_DISTS = ["lognorm", "gamma", "weibull_min", "invgauss", "expon",
                   "fisk", "pearson3", "gengamma", "genextreme", "rayleigh"]

# covariate clusters (for de-clustering at Step 7)
CLUSTERS = {"kinematic": ["speed", "speed_diff"],
            "density":   ["flow", "occupancy", "share_bus"]}

In [2]:
# --- Cell 2: Load data3 and audit integrity (STOP here if anything looks wrong) ---
df = pd.read_excel(DATA_PATH)

SUBJECT_COL = "V_Target" if "V_Target" in df.columns else "V_Subject"
SPEED_COL   = "Target_Speed_km/hr" if "Target_Speed_km/hr" in df.columns else "Subject_Speed_km/hr"
BUS_COL     = next((c for c in df.columns if "bus" in c.lower()), None)

print(f"rows={len(df)}  subject_col={SUBJECT_COL}  speed_col={SPEED_COL}  share_bus={BUS_COL}")
print("total missing cells:", int(df.isna().sum().sum()))
print("\nPair sizes and fit-viability flag:")
vc = df["Pair"].value_counts().sort_values(ascending=False)
for p, n in vc.items():
    flag = "ok" if n >= MIN_N_FIT else ("small" if n >= 15 else "TOO SMALL")
    print(f"  {p:24s} n={n:4d}  [{flag}]")

# covariate registry (dynamic on presence)
REGISTRY = {
    "speed":      dict(col=SPEED_COL,            type="cont", unit=1,    label="per +1 km/h"),
    "speed_diff": dict(col="Speed_Difference",   type="cont", unit=1,    label="per +1 km/h"),
    "flow":       dict(col="Flow_pcu/hr",        type="cont", unit=1000, label="per +1000 pcu/hr"),
    "off_cen":    dict(col="Off_centeredness",   type="bin",  unit=1,    label="True vs False"),
    "occupancy":  dict(col="Occupancy",          type="bin",  unit=1,    label="True vs False"),
}
if BUS_COL is not None:
    s = pd.to_numeric(df[BUS_COL], errors="coerce")
    REGISTRY["share_bus"] = (dict(col=BUS_COL, type="cont", unit=0.1,  label="per +0.10 share")
                             if s.max() <= 1.5 else
                             dict(col=BUS_COL, type="cont", unit=10.0, label="per +10 (pct pts)"))
COVARIATES = {k: v for k, v in REGISTRY.items() if v["col"] in df.columns}
print("\nCovariates available:", list(COVARIATES.keys()))

rows=898  subject_col=V_Target  speed_col=Target_Speed_km/hr  share_bus=None
total missing cells: 0

Pair sizes and fit-viability flag:
  BTW_following_4W         n= 250  [ok]
  BTW_following_MT_3W      n= 186  [ok]
  BTW_following_NMT_3W     n= 119  [ok]
  PR_following_MT_3W       n= 104  [ok]
  BTW_following_MT_2W      n=  73  [ok]
  PR_following_NMT_3W      n=  49  [ok]
  PR_following_4W          n=  43  [ok]
  BTW_following_NMT_2W     n=  41  [ok]
  PR_following_MT_2W       n=  23  [small]
  PR_following_NMT_2W      n=  10  [TOO SMALL]

Covariates available: ['speed', 'speed_diff', 'flow', 'off_cen', 'occupancy']


In [3]:
# --- Cell 3: Non-parametric association helpers (Step 1) ---
def rank_biserial(u, n1, n2): return 1 - (2*u)/(n1*n2)
def epsilon_squared(h, n):    return h/(n-1)
def bh_adjust(pvals):
    p = np.asarray(pvals, float); n = len(p); order = np.argsort(p)
    ranked = np.minimum.accumulate((p[order]*n/np.arange(1, n+1))[::-1])[::-1]
    adj = np.empty(n); adj[order] = np.clip(ranked, 0, 1); return adj

def association_table(data, variables):
    rows = []
    for var, scale in variables:
        sub = data[[var, OUTCOME]].dropna(); n = len(sub)
        if scale == "cont":
            if sub[var].std() == 0: continue
            rho, p = stats.spearmanr(sub[var], sub[OUTCOME])
            rows.append(dict(Variable=var, Scale="continuous", Test="Spearman",
                             Statistic=round(rho,4), N=n, p_raw=p,
                             EffectSize=round(rho,4), EffectType="rho"))
        elif scale == "bin":
            grp = [g[OUTCOME].values for _, g in sub.groupby(var)]
            if len(grp) != 2 or min(map(len, grp)) < 3: continue
            u, p = stats.mannwhitneyu(grp[0], grp[1], alternative="two-sided")
            r = rank_biserial(u, len(grp[0]), len(grp[1]))
            rows.append(dict(Variable=var, Scale="binary", Test="Mann-Whitney U",
                             Statistic=round(u,2), N=n, p_raw=p,
                             EffectSize=round(r,4), EffectType="rank-biserial"))
        else:  # multi
            grp = [g[OUTCOME].values for _, g in sub.groupby(var)]
            if len(grp) < 2: continue
            h, p = stats.kruskal(*grp); e = epsilon_squared(h, n)
            rows.append(dict(Variable=var, Scale=f"multi({sub[var].nunique()})",
                             Test="Kruskal-Wallis", Statistic=round(h,3), N=n, p_raw=p,
                             EffectSize=round(e,4), EffectType="epsilon^2"))
    res = pd.DataFrame(rows)
    if len(res):
        res["p_BH"] = bh_adjust(res["p_raw"].values)
        res["Significant"] = np.where(res["p_BH"] < ALPHA, "Yes", "No")
        res["p_raw"] = res["p_raw"].map(lambda v: "<0.001" if v < 1e-3 else round(v,4))
        res["p_BH"]  = res["p_BH"].map(lambda v: "<0.001" if v < 1e-3 else round(v,4))
        res = res.reindex(res["EffectSize"].abs().sort_values(ascending=False).index).reset_index(drop=True)
    return res

In [4]:
# --- Cell 4: STEP 1 - Inferential statistics (Overall / BTW / PR) -> Excel 1 ---
base_vars = [("V_Leading_Class","multi"), ("Pair","multi"),
             (SPEED_COL,"cont"), ("Leading_Speed_km/hr","cont"), ("Speed_Difference","cont"),
             ("Off_centeredness","bin"), ("Occupancy","bin"), ("Flow_pcu/hr","cont"), ("Site","bin")]

overall_vars = [(SUBJECT_COL, "bin")] + base_vars     # include subject only overall
assoc_overall = association_table(df, overall_vars)
assoc_btw     = association_table(df[df[SUBJECT_COL]=="BTW"], base_vars)
assoc_pr      = association_table(df[df[SUBJECT_COL]=="PR"],  base_vars)

xl1 = os.path.join(TABLES, "01_inferential_stats.xlsx")
with pd.ExcelWriter(xl1, engine="openpyxl") as xl:
    assoc_overall.to_excel(xl, sheet_name="Overall", index=False)
    assoc_btw.to_excel(xl,     sheet_name="BTW_only", index=False)
    assoc_pr.to_excel(xl,      sheet_name="PR_only",  index=False)
print("Saved Excel 1:", xl1)
assoc_overall

Saved Excel 1: D:\Headway\Tables\01_inferential_stats.xlsx


,Variable,Scale,Test,Statistic,N,p_raw,EffectSize,EffectType,p_BH,Significant
0,V_Target,binary,Mann-Whitney U,40255.0000,898,<0.001,0.4745,rank-biserial,<0.001,Yes
1,Target_Speed_km/hr,continuous,Spearman,-0.3901,898,<0.001,-0.3901,rho,<0.001,Yes
2,Speed_Difference,continuous,Spearman,-0.3404,898,<0.001,-0.3404,rho,<0.001,Yes
3,Pair,multi(10),Kruskal-Wallis,159.2200,898,<0.001,0.1775,epsilon^2,<0.001,Yes
4,Flow_pcu/hr,continuous,Spearman,0.1764,898,<0.001,0.1764,rho,<0.001,Yes
5,Occupancy,binary,Mann-Whitney U,73675.5000,898,<0.001,-0.1697,rank-biserial,<0.001,Yes
6,Site,binary,Mann-Whitney U,70364.0000,898,<0.001,0.1665,rank-biserial,<0.001,Yes
7,Off_centeredness,binary,Mann-Whitney U,98455.0000,898,0.0029,-0.1229,rank-biserial,0.0033,Yes
8,V_Leading_Class,multi(5),Kruskal-Wallis,22.2150,898,<0.001,0.0248,epsilon^2,<0.001,Yes
9,Leading_Speed_km/hr,continuous,Spearman,-0.0194,898,0.5624,-0.0194,rho,0.5624,No


In [5]:
# --- Cell 5: Marginal distribution fit helper (Step 2) ---
def fit_marginal(name, data):
    dist = getattr(stats, name)
    try:
        params = dist.fit(data)                       # loc free (3-param where applicable)
        ll = np.sum(dist.logpdf(data, *params))
        if not np.isfinite(ll): return None
        k, n = len(params), len(data)
        ks, ksp = stats.kstest(data, name, args=params)
        shapes = (dist.shapes.split(",") if dist.shapes else [])
        labels = [s.strip() for s in shapes] + ["loc", "scale"]
        pstr = ", ".join(f"{l}={v:.4f}" for l, v in zip(labels, params))
        return dict(Distribution=name, k=k, LogLik=ll, AIC=2*k-2*ll, BIC=k*np.log(n)-2*ll,
                    KS=ks, KS_p=ksp, Params=pstr)
    except Exception:
        return None

In [6]:
# --- Cell 6: STEP 2 - Fit all candidate distributions to every pair ---
all_fit_rows, best_rows, pair_best = [], [], {}
for pair, g in df.groupby("Pair"):
    t = g[OUTCOME].dropna().values; n = len(t)
    fits = [f for f in (fit_marginal(nm, t) for nm in CANDIDATE_DISTS) if f]
    fits.sort(key=lambda d: d["AIC"])
    for rank, f in enumerate(fits, 1):
        all_fit_rows.append(dict(Pair=pair, N=n, Rank=rank, Distribution=f["Distribution"],
                                 AIC=round(f["AIC"],2), BIC=round(f["BIC"],2),
                                 KS=round(f["KS"],4), KS_p=round(f["KS_p"],4), Params=f["Params"]))
    b = fits[0]; pair_best[pair] = b["Distribution"]
    best_rows.append(dict(Pair=pair, N=n, Best_Distribution=b["Distribution"],
                          AIC=round(b["AIC"],2), KS_p=round(b["KS_p"],4),
                          flag="ok" if n >= MIN_N_FIT else ("small" if n>=15 else "TOO SMALL"),
                          Params=b["Params"]))
all_fits = pd.DataFrame(all_fit_rows)
best_per_pair_dist = pd.DataFrame(best_rows).sort_values("N", ascending=False).reset_index(drop=True)
best_per_pair_dist

,Pair,N,Best_Distribution,AIC,KS_p,flag,Params
0,BTW_following_4W,250,weibull_min,622.44,0.9187,ok,"c=2.4049, loc=0.3652, scale=2.1645"
1,BTW_following_MT_3W,186,weibull_min,440.28,0.9001,ok,"c=1.6347, loc=0.4816, scale=1.5418"
2,BTW_following_NMT_3W,119,gengamma,293.42,0.8286,ok,"a=0.3686, c=3.0339, loc=0.5301, scale=2.5862"
3,PR_following_MT_3W,104,genextreme,280.33,0.6508,ok,"c=0.3302, loc=2.5547, scale=0.9291"
4,BTW_following_MT_2W,73,gengamma,162.79,0.8259,ok,"a=0.4376, c=2.2363, loc=0.6000, scale=2.1461"
5,PR_following_NMT_3W,49,genextreme,142.14,0.6040,ok,"c=0.3738, loc=2.4029, scale=1.0178"
6,PR_following_4W,43,gengamma,129.53,0.5656,ok,"a=0.1499, c=8.7833, loc=0.8197, scale=3.8736"
7,BTW_following_NMT_2W,41,gengamma,70.29,0.0893,ok,"a=0.3735, c=2.0770, loc=0.5667, scale=2.0776"
8,PR_following_MT_2W,23,gengamma,60.70,0.5246,small,"a=0.1402, c=5.9398, loc=1.3333, scale=3.7174"
9,PR_following_NMT_2W,10,gengamma,16.44,0.2439,TOO SMALL,"a=0.2616, c=2.3901, loc=0.6000, scale=4.0779"


In [7]:
# --- Cell 7: STEP 3 - Best overall family across pairs (report all) -> Excel 2 ---
# Rank distributions within each pair by AIC, then sum ranks across pairs with n>=MIN_N_FIT.
viable = all_fits[all_fits["N"] >= MIN_N_FIT]
rank_sum = (viable.groupby("Distribution")["Rank"].agg(["mean","sum","count"])
                  .sort_values("mean").reset_index()
                  .rename(columns={"mean":"mean_rank","sum":"rank_sum","count":"n_pairs"}))
wins = (viable.sort_values("AIC").groupby("Pair").first()["Distribution"]
              .value_counts().rename("n_best").reset_index().rename(columns={"index":"Distribution"}))
family_ranking = rank_sum.merge(wins, on="Distribution", how="left").fillna({"n_best":0})
family_ranking["n_best"] = family_ranking["n_best"].astype(int)

AUTO_TOP = family_ranking.iloc[0]["Distribution"]          # data-driven top (flexible dists win raw fit)
# Default to Weibull: parsimonious, interpretable, never KS-rejected, and it is the
# reference your Step 6 assumes ("pairs that don't tip Weibull"). To use the raw
# data-driven pick instead, set OVERALL_FAMILY = AUTO_TOP.
OVERALL_FAMILY = "weibull_min"
print("Data-driven top by mean AIC-rank:", AUTO_TOP,
      "| Using OVERALL_FAMILY =", OVERALL_FAMILY,
      "(set OVERALL_FAMILY = AUTO_TOP to override)")

xl2 = os.path.join(TABLES, "02_distribution_fits.xlsx")
with pd.ExcelWriter(xl2, engine="openpyxl") as xl:
    best_per_pair_dist.to_excel(xl, sheet_name="Best_per_pair", index=False)
    all_fits.to_excel(xl,           sheet_name="All_fits",     index=False)
    family_ranking.to_excel(xl,     sheet_name="Overall_family_ranking", index=False)
print("Saved Excel 2:", xl2)
family_ranking

Data-driven top by mean AIC-rank: gengamma | Using OVERALL_FAMILY = weibull_min (set OVERALL_FAMILY = AUTO_TOP to override)
Saved Excel 2: D:\Headway\Tables\02_distribution_fits.xlsx


,Distribution,mean_rank,rank_sum,n_pairs,n_best
0,gengamma,1.750,14,8,4
1,weibull_min,2.000,16,8,2
2,pearson3,4.625,37,8,0
3,gamma,4.750,38,8,0
4,genextreme,4.875,39,8,2
5,rayleigh,5.500,44,8,0
6,invgauss,6.500,52,8,0
7,lognorm,6.500,52,8,0
8,fisk,9.125,73,8,0
9,expon,9.375,75,8,0


In [8]:
# --- Cell 8: STEP 4 - Drop low-sample pairs (EDIT THIS LIST IF NEEDED) ---
# NOTE: your Step 4 text said drop BTW_following_MT_2W, but that pair (n=73) is your
# intended NEW inclusion (the bike pair). Default below keeps it and drops the
# low-sample bicycle (NMT_2W) pairs plus PR-follows-bike. Change freely.
DROP_PAIRS = ["PR_following_MT_2W", "PR_following_NMT_2W"]

KEPT_PAIRS = [p for p in df["Pair"].unique() if p not in DROP_PAIRS]
KEPT_PAIRS = (best_per_pair_dist[best_per_pair_dist["Pair"].isin(KEPT_PAIRS)]
              .sort_values("N", ascending=False)["Pair"].tolist())
dfk = df[df["Pair"].isin(KEPT_PAIRS)].copy()
print(f"Dropped ({len(DROP_PAIRS)}):", DROP_PAIRS)
print(f"Kept ({len(KEPT_PAIRS)}):")
for p in KEPT_PAIRS:
    print("  ", p, int((df['Pair']==p).sum()))

Dropped (2): ['PR_following_MT_2W', 'PR_following_NMT_2W']
Kept (8):
   BTW_following_4W 250
   BTW_following_MT_3W 186
   BTW_following_NMT_3W 119
   PR_following_MT_3W 104
   BTW_following_MT_2W 73
   PR_following_NMT_3W 49
   PR_following_4W 43
   BTW_following_NMT_2W 41


In [9]:
# --- Cell 9: Generic AFT machinery (scale ~ covariate; shape shared; loc=0) ---
def robust_min(fn, x0, args):
    best = None
    for m, o in (("Nelder-Mead", {"maxiter":20000,"xatol":1e-8,"fatol":1e-8}),
                 ("Powell",       {"maxiter":20000})):
        r = optimize.minimize(fn, x0, args=args, method=m, options=o)
        if best is None or r.fun < best.fun: best = r
    return best

def _nll_null(p, t, dist, ns):
    lp = dist.logpdf(t, *p[:ns], loc=0, scale=np.exp(p[ns]))
    return -lp.sum() if np.all(np.isfinite(lp)) else 1e12
def _nll_cov(p, t, z, dist, ns):
    lp = dist.logpdf(t, *p[:ns], loc=0, scale=np.exp(p[ns] + p[ns+1]*z))
    return -lp.sum() if np.all(np.isfinite(lp)) else 1e12

def aft_null(dist_name, t):
    dist = getattr(stats, dist_name); ns = dist.numargs
    init = dist.fit(t, floc=0)
    x0 = list(init[:ns]) + [np.log(init[-1])]
    r = robust_min(_nll_null, x0, (t, dist, ns))
    return r, -r.fun, ns

def aft_cov(dist_name, t, z, x0_from_null):
    dist = getattr(stats, dist_name); ns = dist.numargs
    r = robust_min(_nll_cov, list(x0_from_null) + [0.0], (t, z, dist, ns))
    return r, -r.fun

def screen_pair(dist_name, t, g, covariates):
    """Univariate covariate screen for one pair under one distribution."""
    r0, ll0, ns = aft_null(dist_name, t); out = []
    for name, meta in covariates.items():
        raw = pd.to_numeric(g[meta["col"]], errors="coerce").values.astype(float)
        if meta["type"] == "cont":
            sd = np.nanstd(raw); z = raw - np.nanmean(raw); ok = sd > 1e-9
        else:
            sd = np.nan; z = raw
            ok = (len(np.unique(raw))==2 and min((raw==0).sum(),(raw==1).sum())>=MIN_BIN_GROUP)
        if not ok:
            out.append(dict(Covariate=name, Effect="-- low variation --", dAIC=np.nan,
                            LR_p=np.nan, improves_strict="n/a")); continue
        r1, ll1 = aft_cov(dist_name, t, z, r0.x)
        b1 = r1.x[ns+1]; LR = 2*(ll1-ll0); p = stats.chi2.sf(LR, 1); dAIC = LR - 2
        eff = (f"{(np.exp(b1*meta['unit'])-1)*100:+.2f}%  {meta['label']}" if meta["type"]=="cont"
               else f"{(np.exp(b1)-1)*100:+.2f}%  ({meta['label']})")
        out.append(dict(Covariate=name, Type=meta["type"], Effect=eff, coef_b1=round(b1,6),
                        LR_chi2=round(LR,2), LR_p="<0.001" if p<1e-3 else round(p,4),
                        dAIC=round(dAIC,2),
                        improves="Yes" if p<ALPHA else "No",
                        improves_strict="Yes" if (p<ALPHA and dAIC>=DAIC_MIN) else "No",
                        converged=bool(r1.success)))
    return out

In [10]:
# --- Cell 10: STEP 5 - Covariate screen per kept pair under OVERALL_FAMILY -> Excel 3 ---
rows = []
for pair in KEPT_PAIRS:
    g = dfk[dfk["Pair"] == pair]; t = g[OUTCOME].values; n = len(t)
    for rec in screen_pair(OVERALL_FAMILY, t, g, COVARIATES):
        rec.update(Pair=pair, N=n); rows.append(rec)
screen_overall = pd.DataFrame(rows)[
    ["Pair","N","Covariate","Type","Effect","coef_b1","LR_chi2","LR_p","dAIC",
     "improves","improves_strict","converged"]]

order_cov = list(COVARIATES.keys())
num = screen_overall.copy(); num["d"] = pd.to_numeric(num["dAIC"], errors="coerce")
dAIC_matrix = num.pivot(index="Pair", columns="Covariate", values="d").reindex(index=KEPT_PAIRS, columns=order_cov).round(2)

# correlation diagnostic among continuous covariates
cont = {k:v["col"] for k,v in COVARIATES.items() if v["type"]=="cont"}
Cc = dfk[list(cont.values())].apply(pd.to_numeric, errors="coerce"); Cc.columns=list(cont.keys())
corr_overall = Cc.corr(method="spearman").round(3)
wp = []
for pair,g in dfk.groupby("Pair"):
    gg = g[list(cont.values())].apply(pd.to_numeric, errors="coerce"); gg.columns=list(cont.keys())
    ks=list(cont.keys())
    for i in range(len(ks)):
        for j in range(i+1,len(ks)):
            wp.append(dict(Pair=pair,var1=ks[i],var2=ks[j],
                           spearman_r=round(stats.spearmanr(gg[ks[i]],gg[ks[j]],nan_policy="omit")[0],3)))
corr_within = pd.DataFrame(wp)

xl3 = os.path.join(TABLES, "03_covariate_screen_overall_family.xlsx")
with pd.ExcelWriter(xl3, engine="openpyxl") as xl:
    screen_overall.to_excel(xl, sheet_name="Screening_long", index=False)
    dAIC_matrix.to_excel(xl,    sheet_name="dAIC_matrix")
    corr_overall.to_excel(xl,   sheet_name="Corr_overall")
    corr_within.to_excel(xl,    sheet_name="Corr_within_pair", index=False)
print("Saved Excel 3:", xl3, "| family:", OVERALL_FAMILY)
dAIC_matrix

Saved Excel 3: D:\Headway\Tables\03_covariate_screen_overall_family.xlsx | family: weibull_min


Covariate,speed,speed_diff,flow,off_cen,occupancy
Pair,,,,,
BTW_following_4W,26.78,29.76,4.25,1.83,-0.54
BTW_following_MT_3W,34.26,26.81,14.13,-1.41,2.58
BTW_following_NMT_3W,4.11,-1.64,2.59,-1.95,-1.99
PR_following_MT_3W,16.08,1.13,2.91,-1.93,9.22
BTW_following_MT_2W,4.86,5.55,1.91,-1.31,-1.82
PR_following_NMT_3W,-1.95,-0.16,-1.32,-0.66,-1.45
PR_following_4W,-1.99,-1.99,-1.98,0.37,-1.79
BTW_following_NMT_2W,-0.26,6.57,-1.39,-1.77,-1.45


In [11]:
# --- Cell 11: STEP 6 - Covariate screen under each pair's OWN best dist (non-Weibull) -> Excel 4 ---
own_pairs = [p for p in KEPT_PAIRS if pair_best[p] != OVERALL_FAMILY]
print("Kept pairs whose own best distribution != overall family:")
for p in own_pairs: print(f"   {p}: own={pair_best[p]}")

rows = []
for pair in own_pairs:
    g = dfk[dfk["Pair"]==pair]; t = g[OUTCOME].values; n = len(t)
    own = pair_best[pair]
    # baseline AIC comparison own vs overall
    _, ll_own, ns_o = aft_null(own, t);  aic_own = 2*(ns_o+1) - 2*ll_own
    _, ll_ov,  ns_v = aft_null(OVERALL_FAMILY, t); aic_ov = 2*(ns_v+1) - 2*ll_ov
    own_recs = {r["Covariate"]: r for r in screen_pair(own, t, g, COVARIATES)}
    ov_recs  = {r["Covariate"]: r for r in screen_pair(OVERALL_FAMILY, t, g, COVARIATES)}
    for cov in COVARIATES:
        o = own_recs[cov]; v = ov_recs[cov]
        rows.append(dict(Pair=pair, N=n, own_dist=own, Covariate=cov,
                         own_effect=o.get("Effect"), own_dAIC=o.get("dAIC"), own_LR_p=o.get("LR_p"),
                         own_strict=o.get("improves_strict"),
                         weibull_dAIC=v.get("dAIC"), weibull_strict=v.get("improves_strict"),
                         unlocked_by_own_dist="Yes" if (o.get("improves_strict")=="Yes" and
                                                        v.get("improves_strict")=="No") else "No",
                         own_baseline_AIC=round(aic_own,2), overall_baseline_AIC=round(aic_ov,2),
                         own_better_baseline="Yes" if aic_own < aic_ov - DAIC_MIN else "No"))
screen_own = pd.DataFrame(rows) if rows else pd.DataFrame(
    columns=["Pair","N","own_dist","Covariate","own_effect","own_dAIC","own_LR_p","own_strict",
             "weibull_dAIC","weibull_strict","unlocked_by_own_dist","own_baseline_AIC",
             "overall_baseline_AIC","own_better_baseline"])

xl4 = os.path.join(TABLES, "04_covariate_screen_ownbest_dist.xlsx")
with pd.ExcelWriter(xl4, engine="openpyxl") as xl:
    screen_own.to_excel(xl, sheet_name="Own_vs_overall", index=False)
print("Saved Excel 4:", xl4)
screen_own

Kept pairs whose own best distribution != overall family:
   BTW_following_NMT_3W: own=gengamma
   PR_following_MT_3W: own=genextreme
   BTW_following_MT_2W: own=gengamma
   PR_following_NMT_3W: own=genextreme
   PR_following_4W: own=gengamma
   BTW_following_NMT_2W: own=gengamma
Saved Excel 4: D:\Headway\Tables\04_covariate_screen_ownbest_dist.xlsx


,Pair,N,own_dist,Covariate,own_effect,own_dAIC,own_LR_p,own_strict,weibull_dAIC,weibull_strict,unlocked_by_own_dist,own_baseline_AIC,overall_baseline_AIC,own_better_baseline
0,BTW_following_NMT_3W,119,gengamma,speed,-2.31% per +1 km/h,3.98,0.0145,Yes,4.11,Yes,No,299.87,301.90,Yes
1,BTW_following_NMT_3W,119,gengamma,speed_diff,-0.65% per +1 km/h,-1.57,0.5112,No,-1.64,No,No,299.87,301.90,Yes
2,BTW_following_NMT_3W,119,gengamma,flow,+6.09% per +1000 pcu/hr,3.15,0.0232,Yes,2.59,Yes,No,299.87,301.90,Yes
3,BTW_following_NMT_3W,119,gengamma,off_cen,-10.64% (True vs False),-0.78,0.2686,No,-1.95,No,No,299.87,301.90,Yes
4,BTW_following_NMT_3W,119,gengamma,occupancy,-8.50% (True vs False),-1.39,0.4361,No,-1.99,No,No,299.87,301.90,Yes
5,PR_following_MT_3W,104,genextreme,speed,-2.74% per +1 km/h,11.68,<0.001,Yes,16.08,Yes,No,414.57,278.65,No
6,PR_following_MT_3W,104,genextreme,speed_diff,+0.38% per +1 km/h,0.97,0.0846,No,1.13,No,No,414.57,278.65,No
7,PR_following_MT_3W,104,genextreme,flow,+3.15% per +1000 pcu/hr,5.12,0.0076,Yes,2.91,Yes,No,414.57,278.65,No
8,PR_following_MT_3W,104,genextreme,off_cen,-8.88% (True vs False),11.31,<0.001,Yes,-1.93,No,Yes,414.57,278.65,No
9,PR_following_MT_3W,104,genextreme,occupancy,-8.29% (True vs False),10.87,<0.001,Yes,9.22,Yes,No,414.57,278.65,No


In [12]:
# --- Cell 12: STEP 7 - Finalize distribution + covariate set per pair -> Excel 5 ---
def cluster_of(cov):
    for cl, mem in CLUSTERS.items():
        if cov in mem: return cl
    return cov   # unclustered (e.g. off_cen)

num_ov = screen_overall.copy(); num_ov["d"] = pd.to_numeric(num_ov["dAIC"], errors="coerce")
own_lookup = {(r["Pair"], r["Covariate"]): r for _, r in screen_own.iterrows()}

final = []
for pair in KEPT_PAIRS:
    n = int((dfk["Pair"]==pair).sum())
    # choose final distribution: own-best only if it is a *materially* better baseline
    final_dist = OVERALL_FAMILY; note = "overall family"
    if pair in own_pairs:
        ob = screen_own[screen_own["Pair"]==pair]
        if len(ob) and ob.iloc[0]["own_better_baseline"] == "Yes":
            final_dist = pair_best[pair]; note = "own best (materially better baseline)"
        else:
            note = f"own best is {pair_best[pair]} but overall family kept for parsimony"

    # gather covariates passing strict under the chosen distribution
    if final_dist == OVERALL_FAMILY:
        sub = num_ov[(num_ov["Pair"]==pair) & (num_ov["improves_strict"]=="Yes")].copy()
        sub["d_"] = sub["d"]
        passing = sub[["Covariate","d_"]].values.tolist()
    else:
        passing = [[r["Covariate"], pd.to_numeric(r["own_dAIC"], errors="coerce")]
                   for (pp,cv), r in own_lookup.items() if pp==pair and r["own_strict"]=="Yes"]

    # de-cluster: keep the single strongest covariate per cluster
    by_cluster = {}
    for cov, d in passing:
        cl = cluster_of(cov)
        if cl not in by_cluster or d > by_cluster[cl][1]:
            by_cluster[cl] = (cov, d)
    final_set = [c for c, _ in sorted(by_cluster.values(), key=lambda x: -x[1])]

    final.append(dict(Pair=pair, N=n, Final_distribution=final_dist,
                      All_passing=", ".join(c for c,_ in sorted(passing, key=lambda x:-x[1])) or "none",
                      Final_covariate_set=", ".join(final_set) if final_set else "none (plain fit)",
                      note=note))
final_sets = pd.DataFrame(final)

xl5 = os.path.join(TABLES, "05_final_covariate_sets.xlsx")
with pd.ExcelWriter(xl5, engine="openpyxl") as xl:
    final_sets.to_excel(xl, sheet_name="Final_model_spec", index=False)
print("Saved Excel 5:", xl5)
final_sets

Saved Excel 5: D:\Headway\Tables\05_final_covariate_sets.xlsx


,Pair,N,Final_distribution,All_passing,Final_covariate_set,note
0,BTW_following_4W,250,weibull_min,"speed_diff, speed, flow","speed_diff, flow",overall family
1,BTW_following_MT_3W,186,weibull_min,"speed, speed_diff, flow, occupancy","speed, flow",overall family
2,BTW_following_NMT_3W,119,gengamma,"speed, flow","speed, flow",own best (materially better baseline)
3,PR_following_MT_3W,104,weibull_min,"speed, occupancy, flow","speed, occupancy",own best is genextreme but overall family kept...
4,BTW_following_MT_2W,73,weibull_min,"speed_diff, speed",speed_diff,own best is gengamma but overall family kept f...
5,PR_following_NMT_3W,49,weibull_min,none,none (plain fit),own best is genextreme but overall family kept...
6,PR_following_4W,43,weibull_min,none,none (plain fit),own best is gengamma but overall family kept f...
7,BTW_following_NMT_2W,41,weibull_min,speed_diff,speed_diff,own best is gengamma but overall family kept f...
